# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step tutorial for loading, exploring, and processing the [FAIR^2](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library, referencing all data elements by their `@id` as per the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print overview
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers, as defined in the Croissant metadata.

In [ ]:
# List available record sets and their fields (using @id)
record_sets = list(dataset.record_sets)

if record_sets:
    print("Available Record Sets:")
    for rec in record_sets:
        print(f"  - @id: {rec['@id']}")
        print(f"    Name: {rec.get('name','<no name>')}")
        print("    Fields:")
        for fld in rec.get('field', []):
            if isinstance(fld, dict):
                print(f"      * @id: {fld.get('@id')}  |  name: {fld.get('name','<no name>')}  |  dataType: {fld.get('dataType')}")
            else:
                print(f"      * {fld}")
        print()
else:
    print("No record sets found in dataset metadata.")

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis.

Replace the `<record_set_id>` below with the actual `@id` of a record set from the previous overview. All extraction and referencing uses `@id` per FAIR and Croissant best practice.

In [ ]:
# Collect all record set @ids
record_set_ids = [rec['@id'] for rec in dataset.record_sets]

dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded DataFrame for record set '@id': {rec_id}")
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head(2))
        print()
    else:
        print(f"No records found for record set '@id': {rec_id}\n")

# Example: Access the first record set for further analysis
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Selected Record Set for analysis: {selected_record_set_id}")
    print(dataframes[selected_record_set_id].head())
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering numeric fields, normalization, and grouping by categorical fields.

Refer to all fields and columns by their `@id` as presented in the overview above. Adjust the below example as appropriate for your dataset's available fields.

In [ ]:
# Example EDA: Use a numeric field in the selected record set, referenced by @id
import numpy as np

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    # Identify a numeric field by @id (choose one from df.columns as appropriate)
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # For demonstration, use first numeric field
        print(f"Using numeric field '@id': {numeric_field_id}")

        threshold = 10  # Example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records where {numeric_field_id} > {threshold} (showing up to 5 rows):")
        print(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a categorical field if available
        cat_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if cat_fields:
            group_field_id = cat_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nMean of '{numeric_field_id}' grouped by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("\nNo categorical field available for grouping in this record set.")
    else:
        print("No numeric fields found in the selected record set for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize a numeric field distribution or explore a relationship between two fields.
All axes and labels should refer to fields by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouping field exists, show boxplot
    if cat_fields:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore a FAIR Croissant dataset using the `mlcroissant` library referencing all data by their `@id`. You explored the record sets, extracted data to pandas DataFrames, and performed preliminary EDA and visualization steps.

For further analysis, explore field data types, handle missing values, or build models directly referencing the Croissant schema!
